In [1]:
using ProjectRoot
using Pkg

Pkg.activate(@projectroot)
using QuantumToolbox
using Revise

include("../src/XoniumUnitCell.jl")

  Activating project at `~/Projects/xonium/XoniumUnitCell.jl`


fluxonium (generic function with 1 method)

In [ ]:
"""
    ncell(params::NamedTuple; subsys_nstate=10, diagonalize_subsys=true, set_energy_zero=true, etol=1e-9) -> CompositeQuantumSystem
Construct a composite quantum system representing a unit cell of `fluxonium` qubits, given a `NamedTuple` of parameters for each subsystem.
The function allows for optional diagonalization and truncation of each subsystem.
# Arguments
- `params::NamedTuple`: A `NamedTuple` where each key is a symbol representing a subsystem name, and each value is a `NamedTuple` of
   parameters for the `fluxonium` constructor.
- `subsys_nstate::Int`: The number of states to retain in each subsystem after truncation (default: 10).
- `diagonalize_subsys::Bool`: Whether to diagonalize each subsystem before truncation (default: true).
- `set_energy_zero::Bool`: Whether to set the ground state energy to zero when diagonalizing (default: true).
- `etol::Float64`: The energy tolerance for truncation when diagonalizing (default: 1e-9).
# Returns
- `CompositeQuantumSystem`: A composite quantum system representing the unit cell, with each subsystem constructed according to the 
   provided parameters and optionally diagonalized and truncated.
"""
function ncell(
    params::NamedTuple;
    subsys_nstate=10,
    diagonalize_subsys=true,
    set_energy_zero=true,
    etol=1e-9,)::CompositeQuantumSystem
    return mapreduce(⊗, keys(params), values(params)) do n, p 
        fluxonium(;p..., name=n) |>
            sys -> diagonalize_subsys ? to_energy_basis(sys; set_energy_zero, etol) : sys |>
            sys -> truncate(sys, subsys_nstate)
    end
end

## 2-Cell

In [ ]:
flA_params = (;EJ = 7.0, EC=1.0, EL=1.0, δϕ=0.0, n_g=0.0)
flB_params = (;EJ = 7.0, EC=1.0, EL=0.89, δϕ=0.0, n_g=0.0)
two_cell_params = (flA=flA_params, flB=flB_params)
two_cell = ncell(two_cell_params)

(0.0 + 0.0im, 
Quantum Object:   type=Ket()   dims=([40, 40], [1])   size=(1600,)
1600-element Vector{ComplexF64}:
 0.0 + 0.0im
 1.0 + 0.0im
 0.0 + 0.0im
 0.0 + 0.0im
 0.0 + 0.0im
 0.0 + 0.0im
 0.0 + 0.0im
 0.0 + 0.0im
 0.0 + 0.0im
 0.0 + 0.0im
     ⋮
 0.0 + 0.0im
 0.0 + 0.0im
 0.0 + 0.0im
 0.0 + 0.0im
 0.0 + 0.0im
 0.0 + 0.0im
 0.0 + 0.0im
 0.0 + 0.0im
 0.0 + 0.0im)

In [ ]:
map(keys(two_cell.subsys), values(two_cell.subsys)) do nm, sys 
    (nm, sys)
end

((:flA, SimpleQuantumSystem{@NamedTuple{φ::QuantumObject{Operator, Dimensions{Space, Space}, Matrix{ComplexF64}}, n::QuantumObject{Operator, Dimensions{Space, Space}, Matrix{ComplexF64}}, H::QuantumObject{Operator, Dimensions{Space, Space}, Matrix{ComplexF64}}}, @NamedTuple{EC::Float64, EL::Float64, EJ::Float64}}((φ = 
Quantum Object:   type=Operator()   dims=([10], [10])   size=(10, 10)   ishermitian=false
10×10 Matrix{ComplexF64}:
    -3.14159+0.0im      2.59324+0.0im  …   -0.0010438+0.0im
     2.59324+0.0im     -3.14159+0.0im     -3.07128e-7+0.0im
 -3.16467e-7+0.0im     0.895276+0.0im      0.00685532+0.0im
    0.668003+0.0im   3.48555e-7+0.0im      1.26659e-6+0.0im
 -1.53389e-8+0.0im    -0.189446+0.0im      -0.0539166+0.0im
  -0.0300856+0.0im   1.74424e-7+0.0im  …   -4.9236e-6+0.0im
  -3.3859e-8+0.0im   -0.0144796+0.0im      -0.0750704+0.0im
  -0.0127032+0.0im  -4.82476e-7+0.0im     -1.34562e-5+0.0im
 -2.18671e-7+0.0im   0.00414181+0.0im         3.33669+0.0im
  -0.0010438+0.0im  -3.

In [5]:
two_cell = flA ⊗ flB
print(keys(two_cell.subsys))
two_cell = two_cell + two_cell[:flA, :n, :flB, :n] * 1e-3
two_cell = to_energy_basis(two_cell; set_energy_zero=true, etol=1e-9) # 1Hz error cutoff
state(two_cell, :ge, true)

(:flA, :flB)

(0.0 + 0.0im, 
Quantum Object:   type=Ket()   dims=([10, 10], [1])   size=(100,)
100-element Vector{ComplexF64}:
   0.037998810545775585 + 0.0im
 -1.6459268316834025e-8 + 0.0im
   -5.30034354359328e-9 + 0.0im
 -2.2768147027547425e-7 + 0.0im
  -0.002646755563886421 + 0.0im
      0.365153743081511 + 0.0im
  -6.058479767163695e-6 + 0.0im
   4.983786192943859e-5 + 0.0im
   3.150206821668096e-5 + 0.0im
       -0.2134661497455 + 0.0im
                        ⋮
                    0.0 + 0.0im
                    0.0 + 0.0im
                    0.0 + 0.0im
                    0.0 + 0.0im
                    0.0 + 0.0im
                    0.0 + 0.0im
                    0.0 + 0.0im
                    0.0 + 0.0im
                    0.0 + 0.0im)